In [2]:
import geopandas as gpd
import pandas as pd

In [3]:
hotspots = gpd.read_file('../data/analysis/all_hotspots_2023_2025_night_nh_scan_track.geojson')
hotspots['datetime'] = pd.to_datetime(hotspots['acq_date'])

hotspots['doy'] = pd.to_datetime(hotspots['acq_date']).dt.dayofyear
hotspots = hotspots.to_crs('EPSG:3005')


# split by year
hotspots_2023 = hotspots[hotspots['year'] == 2023].copy()
hotspots_2024 = hotspots[hotspots['year'] == 2024].copy()
hotspots_2025 = hotspots[hotspots['year'] == 2025].copy()

In [4]:
# SDD lookup per fire zone and year
sdd_lookup = {
    'Fort St. John': {2024: 135.71, 2025: 140.63},
    'Fort Nelson':   {2024: 165.14, 2025: 167.72}
}

# fall: after Aug 1 (DOY 213) — captures last hotspots before winter dormancy
# spring: after SDD DOY — captures first hotspots after snowmelt
# using the later of the two fire zones as a conservative cutoff for splitting
# individual fire-level SDD filtering happens in step 3

FALL_DOY_START = 213  # Aug 1

def split_hotspots_fall_spring(hotspots_year, year, sdd_lookup):
    """
    Split annual hotspots into fall and spring subsets.
    Spring cutoff uses the earliest SDD across fire zones as a 
    conservative lower bound — per-fire SDD filtering applied later.
    """
    fall = hotspots_year[hotspots_year['doy'] >= FALL_DOY_START].copy()

    # only compute spring if year exists in sdd_lookup
    sdd_values = [sdd_lookup[zone][year] for zone in sdd_lookup if year in sdd_lookup[zone]]
    if sdd_values:
        min_sdd = min(sdd_values)
        spring = hotspots_year[hotspots_year['doy'] >= min_sdd].copy()
    else:
        print(f"No SDD found for year {year}, spring subset will be empty")
        spring = gpd.GeoDataFrame(columns=hotspots_year.columns, crs=hotspots_year.crs)

    return fall, spring

fall_2023, _            = split_hotspots_fall_spring(hotspots_2023, 2023, sdd_lookup)  # fall only, no spring needed for 2023
fall_2024, spring_2024  = split_hotspots_fall_spring(hotspots_2024, 2024, sdd_lookup)
fall_2025, spring_2025  = split_hotspots_fall_spring(hotspots_2025, 2025, sdd_lookup)  # spring only, no fall needed for 2025

No SDD found for year 2023, spring subset will be empty


In [5]:
def detect_overwintering(fires, buffered_dfs, fall_hotspots, spring_hotspots,
                         buffer_dist, fire_year, spring_year):
    """
    For a given fire year and spring year, find fires where:
    1. Fall hotspots exist within the fire perimeter before winter
    2. Spring hotspots exist within the buffer after per-fire per-buffer SDD
    """

    perims = fires[fires['fire_year'] == float(fire_year)].copy()
    buffers = buffered_dfs[buffer_dist][buffered_dfs[buffer_dist]['fire_year'] == float(fire_year)].copy()

    results = []

    for _, fire in perims.iterrows():
        fid = fire['fire_id']
        fire_zone = fire['fire_zone']
        fire_geom = fire['geometry']

        # get buffer geometry and SDD for this fire
        buf_row = buffers[buffers['fire_id'] == fid]
        if buf_row.empty:
            print(f"No buffer found for {fid}, skipping")
            continue
        buf_row = buf_row.iloc[0]
        buf_geom = buf_row['geometry']

        # pull per-buffer SDD mean for the spring year
        sdd_col = f'sdd_{spring_year}_mean'
        if sdd_col not in buf_row.index or pd.isna(buf_row[sdd_col]):
            print(f"No SDD value for {fid} at {buffer_dist}m buffer for {spring_year}, skipping")
            continue
        sdd_doy = buf_row[sdd_col]

        # --- fall hotspots within perimeter ---
        fall_in_perim = fall_hotspots[fall_hotspots.within(fire_geom)]

        # --- spring hotspots within buffer after per-fire SDD ---
        spring_after_sdd = spring_hotspots[spring_hotspots['doy'] >= sdd_doy]
        spring_in_buffer = spring_after_sdd[spring_after_sdd.within(buf_geom)]

        # classify
        has_fall   = len(fall_in_perim) >= 2
        has_spring = len(spring_in_buffer) > 0
        overwintered = has_fall and has_spring

        results.append({
            'fire_id':              fid,
            'fire_year':            fire_year,
            'fire_zone':            fire_zone,
            'buffer_dist':          buffer_dist,
            'spring_year':          spring_year,
            'sdd_doy':              sdd_doy,  # now per-fire per-buffer rather than zone average
            'n_fall_hotspots':      len(fall_in_perim),
            'n_spring_hotspots':    len(spring_in_buffer),
            'has_fall_hotspots':    has_fall,
            'has_spring_hotspots':  has_spring,
            'overwintered':         overwintered,
            'last_fall_date':       fall_in_perim['datetime'].max() if has_fall else None,
            'last_fall_lat':        fall_in_perim.loc[fall_in_perim['datetime'].idxmax(), 'latitude'] if has_fall else None,
            'last_fall_lon':        fall_in_perim.loc[fall_in_perim['datetime'].idxmax(), 'longitude'] if has_fall else None,
            'first_spring_date':    spring_in_buffer['datetime'].min() if has_spring else None,
            'first_spring_lat':     spring_in_buffer.loc[spring_in_buffer['datetime'].idxmin(), 'latitude'] if has_spring else None,
            'first_spring_lon':     spring_in_buffer.loc[spring_in_buffer['datetime'].idxmin(), 'longitude'] if has_spring else None,
        })

    return results



In [6]:
# fire perimeters with all attributes
fires = gpd.read_file('../data/analysis/all_fires_processed.geojson')
# needs: fire_id, fire_year, fire_zone, geometry

# buffered perimeters per distance with SDD columns attached
buffered_dfs = {
    500:  gpd.read_file('../data/analysis/fires_buffer_500m_with_sdd.geojson'),
    1000: gpd.read_file('../data/analysis/fires_buffer_1000m_with_sdd.geojson'),
    2000: gpd.read_file('../data/analysis/fires_buffer_2000m_with_sdd.geojson')
}
# needs: fire_id, fire_year, fire_zone, geometry, sdd_2024_mean, sdd_2025_mean

buffer_distances = [500, 1000, 2000]

# sanity check
print(f"fires: {len(fires)} | columns: {fires.columns.tolist()}")
print(f"fires CRS: {fires.crs}")
print(f"fires year unique: {fires['fire_year'].unique()}")

for dist in buffer_distances:
    gdf = buffered_dfs[dist]
    print(f"\nbuffer {dist}m: {len(gdf)} | columns: {gdf.columns.tolist()}")
    print(f"  SDD columns present: {[c for c in gdf.columns if 'sdd' in c]}")

fires: 205 | columns: ['fire_no', 'version_no', 'fire_year', 'fire_cause', 'firelabel', 'size_ha', 'source', 'track_date', 'load_date', 'fire_date', 'method', 'fcode', 'shape', 'objectid', 'area_sqm', 'feat_len', 'fire_stat', 'fire_url', 'fire_id', 'fire_zone', 'fire_month', 'fire_doy', 'geometry_wkt', 'overlap', 'overlap_year', 'overlap_id', 'geometry']
fires CRS: EPSG:3005
fires year unique: [2024 2023 2025]

buffer 500m: 141 | columns: ['fire_id', 'fire_year', 'sdd_2024_mean', 'sdd_2024_min', 'sdd_2024_max', 'sdd_2025_mean', 'sdd_2025_min', 'sdd_2025_max', 'geometry']
  SDD columns present: ['sdd_2024_mean', 'sdd_2024_min', 'sdd_2024_max', 'sdd_2025_mean', 'sdd_2025_min', 'sdd_2025_max']

buffer 1000m: 141 | columns: ['fire_id', 'fire_year', 'sdd_2024_mean', 'sdd_2024_min', 'sdd_2024_max', 'sdd_2025_mean', 'sdd_2025_min', 'sdd_2025_max', 'geometry']
  SDD columns present: ['sdd_2024_mean', 'sdd_2024_min', 'sdd_2024_max', 'sdd_2025_mean', 'sdd_2025_min', 'sdd_2025_max']

buffer 2000m

In [7]:

# --- run for all buffer distances and year combinations ---
all_results = {}

for dist in buffer_distances:
    results_23_24 = detect_overwintering(
        fires, buffered_dfs, fall_2023, spring_2024,
        dist, fire_year=2023, spring_year=2024
    )

    results_24_25 = detect_overwintering(
        fires, buffered_dfs, fall_2024, spring_2025,
        dist, fire_year=2024, spring_year=2025
    )

    df_23_24 = pd.DataFrame(results_23_24)
    df_24_25 = pd.DataFrame(results_24_25)

    all_results[dist] = pd.concat([df_23_24, df_24_25], ignore_index=True)

    print(f"\nBuffer {dist}m:")
    print(f"  2023->2024 overwintering: {df_23_24['overwintered'].sum()} of {len(df_23_24)} fires")
    print(f"  2024->2025 overwintering: {df_24_25['overwintered'].sum()} of {len(df_24_25)} fires")


Buffer 500m:
  2023->2024 overwintering: 14 of 80 fires
  2024->2025 overwintering: 4 of 61 fires

Buffer 1000m:
  2023->2024 overwintering: 14 of 80 fires
  2024->2025 overwintering: 4 of 61 fires

Buffer 2000m:
  2023->2024 overwintering: 16 of 80 fires
  2024->2025 overwintering: 4 of 61 fires


In [8]:
# if not in memory, reload
multiyear_dfs = {}
for dist in buffer_distances:
    path = f"../data/analysis/multiyear_overwintering_{dist}m.csv"
    multiyear_dfs[dist] = pd.read_csv(path)
    print(f"Buffer {dist}m — multiyear_dfs: {len(multiyear_dfs[dist])} rows | columns: {multiyear_dfs[dist].columns.tolist()}")

Buffer 500m — multiyear_dfs: 10 rows | columns: ['fire_id_2023', 'fire_id_2024', 'fire_zone', 'n_fall_hotspots_2023', 'n_spring_hotspots_2024', 'hotspot_distance_m_2324', 'dormancy_days_2324', 'days_after_sdd_2324', 'dist_to_perimeter_2324', 'last_fall_date_2023', 'first_spring_date_2024', 'n_fall_hotspots_2024', 'n_spring_hotspots_2025', 'hotspot_distance_m_2425', 'dormancy_days_2425', 'days_after_sdd_2425', 'dist_to_perimeter_2425', 'last_fall_date_2024', 'first_spring_date_2025', 'fire_id_2025', 'buffer_dist']
Buffer 1000m — multiyear_dfs: 11 rows | columns: ['fire_id_2023', 'fire_id_2024', 'fire_zone', 'n_fall_hotspots_2023', 'n_spring_hotspots_2024', 'hotspot_distance_m_2324', 'dormancy_days_2324', 'days_after_sdd_2324', 'dist_to_perimeter_2324', 'last_fall_date_2023', 'first_spring_date_2024', 'n_fall_hotspots_2024', 'n_spring_hotspots_2025', 'hotspot_distance_m_2425', 'dormancy_days_2425', 'days_after_sdd_2425', 'dist_to_perimeter_2425', 'last_fall_date_2024', 'first_spring_date

In [9]:
multiyear_results = {}

for dist in buffer_distances:
    df = all_results[dist]

    # fires that overwintered 2023->2024
    ow_23_24 = set(df[(df['fire_year'] == 2023) & (df['overwintered'])]['fire_id'])

    # fires that overwintered 2024->2025
    ow_24_25 = set(df[(df['fire_year'] == 2024) & (df['overwintered'])]['fire_id'])

    # cross reference using multiyear_dfs (perimeter overlap chains built earlier)
    if dist in multiyear_dfs and not multiyear_dfs[dist].empty:
        mdf = multiyear_dfs[dist].copy()

        # keep chains where 2023 fire overwintered into 2024
        # AND the linked 2024 fire overwintered into 2025
        mdf['overwinter_23_24'] = mdf['fire_id_2023'].isin(ow_23_24)
        mdf['overwinter_24_25'] = mdf['fire_id_2024'].isin(ow_24_25)
        mdf['multiyear'] = mdf['overwinter_23_24'] & mdf['overwinter_24_25']

        multiyear_results[dist] = mdf[mdf['multiyear']].copy()

        print(f"\nBuffer {dist}m:")
        print(f"  2023 fires that overwintered into 2024: {len(ow_23_24)}")
        print(f"  2024 fires that overwintered into 2025: {len(ow_24_25)}")
        print(f"  confirmed multi-year (2023->2024->2025): {len(multiyear_results[dist])}")
        print(multiyear_results[dist][['fire_id_2023', 'fire_id_2024', 'fire_id_2025']])

    else:
        print(f"\nBuffer {dist}m — no perimeter overlap chains found, cannot confirm multi-year fires")
        multiyear_results[dist] = pd.DataFrame()

# save
for dist in buffer_distances:
    if not multiyear_results[dist].empty:
        multiyear_results[dist].to_csv(f"../data/analysis/multiyear_results_{dist}m.csv", index=False)
        print(f"Saved multiyear_results_{dist}m.csv")


Buffer 500m:
  2023 fires that overwintered into 2024: 14
  2024 fires that overwintered into 2025: 4
  confirmed multi-year (2023->2024->2025): 10
  fire_id_2023 fire_id_2024     fire_id_2025
0  2023_G90288  2024_G90228  ['2025_G80320']
1  2023_G90288  2024_G90686  ['2025_G90216']
2  2023_G90628  2024_G90228  ['2025_G80320']
3  2023_G91313  2024_G90228  ['2025_G80320']
4  2023_G91494  2024_G90228  ['2025_G80320']
5  2023_G91534  2024_G90228  ['2025_G80320']
6  2023_G91534  2024_G90399  ['2025_G90382']
7  2023_G92498  2024_G90228  ['2025_G80320']
8  2023_G92940  2024_G90228  ['2025_G80320']
9  2023_G93291  2024_G90228  ['2025_G80320']

Buffer 1000m:
  2023 fires that overwintered into 2024: 14
  2024 fires that overwintered into 2025: 4
  confirmed multi-year (2023->2024->2025): 11
   fire_id_2023 fire_id_2024     fire_id_2025
0   2023_G90288  2024_G90228  ['2025_G80320']
1   2023_G90288  2024_G90399  ['2025_G90400']
2   2023_G90288  2024_G90686  ['2025_G90216']
3   2023_G90628  2024_

In [10]:
import ast

# fix fire_year dtype in all_results
for dist in buffer_distances:
    all_results[dist]['fire_year'] = all_results[dist]['fire_year'].astype(int)

# fix stringified lists in multiyear_dfs
for dist in buffer_distances:
    multiyear_dfs[dist]['fire_id_2025'] = multiyear_dfs[dist]['fire_id_2025'].apply(
        lambda x: ast.literal_eval(x) if isinstance(x, str) else x
    )
    print(f"Buffer {dist}m multiyear_dfs fixed:")
    print(multiyear_dfs[dist].head())

Buffer 500m multiyear_dfs fixed:
  fire_id_2023 fire_id_2024              fire_zone  n_fall_hotspots_2023  \
0  2023_G90288  2024_G90228  Fort Nelson Fire Zone                  1227   
1  2023_G90288  2024_G90686  Fort Nelson Fire Zone                  1227   
2  2023_G90628  2024_G90228                   both                  2730   
3  2023_G91313  2024_G90228  Fort Nelson Fire Zone                  2267   
4  2023_G91494  2024_G90228  Fort Nelson Fire Zone                   256   

   n_spring_hotspots_2024  hotspot_distance_m_2324  dormancy_days_2324  \
0                     292             50271.083503               298.0   
1                     292             50271.083503               298.0   
2                     132             24707.954827               272.0   
3                      14              9943.502766               269.0   
4                      16              6863.841559               302.0   

   days_after_sdd_2324  dist_to_perimeter_2324 last_fall_date_202

In [11]:
for dist in buffer_distances:
    df = all_results[dist]
    ow = df[df['overwintered'] == True]
    print(f"\nBuffer {dist}m — overwintered fires: {len(ow)}")
    print(f"Columns: {df.columns.tolist()}")
    print(ow[['fire_id', 'fire_year', 'spring_year', 
               'last_fall_date', 'last_fall_lat', 'last_fall_lon',
               'first_spring_date', 'first_spring_lat', 'first_spring_lon']].head())


Buffer 500m — overwintered fires: 18
Columns: ['fire_id', 'fire_year', 'fire_zone', 'buffer_dist', 'spring_year', 'sdd_doy', 'n_fall_hotspots', 'n_spring_hotspots', 'has_fall_hotspots', 'has_spring_hotspots', 'overwintered', 'last_fall_date', 'last_fall_lat', 'last_fall_lon', 'first_spring_date', 'first_spring_lat', 'first_spring_lon']
        fire_id  fire_year  spring_year last_fall_date  last_fall_lat  \
17  2023_G80280       2023         2024     2023-12-14       57.10903   
18  2023_G80291       2023         2024     2023-09-23       56.62333   
37  2023_G90207       2023         2024     2023-09-25       59.31479   
41  2023_G90288       2023         2024     2023-09-11       59.67613   
44  2023_G90628       2023         2024     2023-09-23       58.27380   

    last_fall_lon first_spring_date  first_spring_lat  first_spring_lon  
17     -121.83142        2024-07-10          57.23188        -122.22099  
18     -121.21746        2024-05-25          56.58251        -121.25781  


# Step 7 - Calculate overwintering metrics

In [12]:
from pyproj import Transformer
from scipy.spatial.distance import euclidean
from shapely.geometry import Point

# transformer from lat/lon to BC Albers for distance calculation
transformer = Transformer.from_crs('EPSG:4326', 'EPSG:3005', always_xy=True)

def calculate_overwintering_metrics(all_results, fires, buffer_distances):
    """
    For each overwintering fire calculate:
    - euclidean distance between last fall and first spring hotspot
    - dormancy days between last fall and first spring hotspot
    - days after SDD of first spring hotspot
    - distance from first spring hotspot to previous year's perimeter edge
    """
    for dist in buffer_distances:
        df = all_results[dist].copy()

        # ensure datetime columns are datetime type
        df['last_fall_date']    = pd.to_datetime(df['last_fall_date'])
        df['first_spring_date'] = pd.to_datetime(df['first_spring_date'])

        # only calculate for overwintering fires
        mask = df['overwintered'] == True

        # project lat/lon to EPSG:3005 for accurate distance
        last_fall_x, last_fall_y = transformer.transform(
            df.loc[mask, 'last_fall_lon'].values,
            df.loc[mask, 'last_fall_lat'].values
        )
        first_spring_x, first_spring_y = transformer.transform(
            df.loc[mask, 'first_spring_lon'].values,
            df.loc[mask, 'first_spring_lat'].values
        )

        # euclidean distance in metres between last fall and first spring hotspot
        df.loc[mask, 'hotspot_distance_m'] = [
            euclidean([x1, y1], [x2, y2])
            for x1, y1, x2, y2 in zip(last_fall_x, last_fall_y, first_spring_x, first_spring_y)
        ]

        # dormancy duration in days
        df.loc[mask, 'dormancy_days'] = (
            df.loc[mask, 'first_spring_date'] - df.loc[mask, 'last_fall_date']
        ).dt.days

        # days after SDD of first spring hotspot
        df.loc[mask, 'first_spring_doy'] = df.loc[mask, 'first_spring_date'].dt.dayofyear
        df.loc[mask, 'days_after_sdd'] = df.loc[mask, 'first_spring_doy'] - df.loc[mask, 'sdd_doy']

        # distance from first spring hotspot to previous year perimeter edge
        dist_to_perim = []
        for _, row in df[mask].iterrows():
            perim = fires[fires['fire_id'] == row['fire_id']]
            if perim.empty:
                dist_to_perim.append(None)
                continue

            perim_geom = perim.iloc[0]['geometry']

            spring_point = Point(
                first_spring_x[list(df[mask].index).index(_.name if hasattr(_, 'name') else _)],
                first_spring_y[list(df[mask].index).index(_.name if hasattr(_, 'name') else _)]
            )

            dist_to_perim.append(spring_point.distance(perim_geom.boundary))

        df.loc[mask, 'dist_to_prev_perimeter_m'] = dist_to_perim

        all_results[dist] = df

        # review
        print(f"\nBuffer {dist}m — overwintering metrics:")
        cols = ['fire_id', 'fire_year', 'spring_year',
                'hotspot_distance_m', 'dormancy_days', 
                'days_after_sdd', 'dist_to_prev_perimeter_m']
        print(df[mask][cols].to_string())

    return all_results

all_results = calculate_overwintering_metrics(all_results, fires, buffer_distances)

# save
for dist in buffer_distances:
    all_results[dist].to_csv(f"../data/analysis/overwintering_results_{dist}m.csv", index=False)
    print(f"Saved overwintering_results_{dist}m.csv")


Buffer 500m — overwintering metrics:
         fire_id  fire_year  spring_year  hotspot_distance_m  dormancy_days  days_after_sdd  dist_to_prev_perimeter_m
17   2023_G80280       2023         2024        27228.818867          209.0       46.385802               1294.976745
18   2023_G80291       2023         2024         5182.777189          245.0       37.738742               4956.134616
37   2023_G90207       2023         2024        41151.116522          233.0        5.694656               2748.749950
41   2023_G90288       2023         2024        50271.083503          298.0       56.528890                454.074048
44   2023_G90628       2023         2024        24707.954827          272.0       50.618965                320.561076
56   2023_G91313       2023         2024         9943.502766          269.0       46.262006                634.109224
59   2023_G91467       2023         2024         1390.445743          267.0       13.560606                368.405727
60   2023_G91494  

In [14]:
for dist in buffer_distances:
    all_results[dist].to_csv(f"../data/analysis/overwintering_results_{dist}m.csv", index=False)
    print(f"Saved overwintering_results_{dist}m.csv")

Saved overwintering_results_500m.csv
Saved overwintering_results_1000m.csv
Saved overwintering_results_2000m.csv


# Step 8 - summary statistics
per zone / year

In [15]:
def calculate_summary_statistics(all_results, buffer_distances):
    """
    Summary statistics per fire zone and year:
    - number of overwintering fires
    - number of non-overwintering fires
    - number of multi-year overwintering fires
    - average distance between latest and earliest hotspots
    - average time difference between latest and earliest hotspots
    - average distance of spring hotspot to previous year's perimeter
    - average time difference between spring hotspot and snowmelt onset day
    """

    for dist in buffer_distances:
        df = all_results[dist].copy()

        summary = df.groupby(['fire_zone', 'spring_year']).agg(
            n_total=('fire_id', 'count'),
            n_overwintering=('overwintered', 'sum'),
            n_non_overwintering=('overwintered', lambda x: (~x).sum()),
            avg_hotspot_distance_m=('hotspot_distance_m', 'mean'),
            avg_dormancy_days=('dormancy_days', 'mean'),
            avg_days_after_sdd=('days_after_sdd', 'mean'),
        ).reset_index()

        # add multi-year counts per fire zone
        if dist in multiyear_results and not multiyear_results[dist].empty:
            # join fire_zone from fires onto multiyear_results
            mdf = multiyear_results[dist].merge(
                fires[['fire_id', 'fire_zone']].rename(columns={'fire_id': 'fire_id_2023'}),
                on='fire_id_2023',
                how='left'
            )
            multiyear_counts = mdf.groupby('fire_zone').size().reset_index(name='n_multiyear')
            summary = summary.merge(multiyear_counts, on='fire_zone', how='left')
            summary['n_multiyear'] = summary['n_multiyear'].fillna(0).astype(int)
        else:
            summary['n_multiyear'] = 0

        print(f"\nBuffer {dist}m — summary statistics:")
        print(summary.to_string())

        # save
        summary.to_csv(f"../data/analysis/summary_statistics_{dist}m.csv", index=False)
        print(f"Saved summary_statistics_{dist}m.csv")

calculate_summary_statistics(all_results, buffer_distances)

KeyError: 'fire_zone'

# Step 8 - multi-year overwintering

In [22]:
def match_hotspot_to_nearest_perimeter(hotspot_gdf, perims, id_col_left, id_col_right, max_dist_m=500):
    """
    Match each hotspot to the nearest perimeter within max_dist_m.
    Uses distance-based matching rather than within/intersects.
    """
    records = []
    for _, row in hotspot_gdf.iterrows():
        pt = row['geometry']
        perims_copy = perims.copy()
        perims_copy['dist'] = perims_copy['geometry'].distance(pt)
        nearest = perims_copy[perims_copy['dist'] <= max_dist_m]
        if not nearest.empty:
            for _, perim_row in nearest.iterrows():
                records.append({
                    id_col_left:  row[id_col_left],
                    id_col_right: perim_row[id_col_right],
                    'dist_to_perimeter': perim_row['dist']
                })
    return pd.DataFrame(records)

In [23]:
def find_multiyear_overwintering(all_results, fires, buffer_distances, hotspot_buffer_m=500):
    multiyear_results = {}

    for dist in buffer_distances:
        df = all_results[dist].copy()

        # --- step 1: 2023 fires with spring 2024 hotspots in 2023 buffer ---
        ow_23_24 = df[
            (df['fire_year'] == 2023) &
            (df['overwintered'] == True) &
            (df['spring_year'] == 2024)
        ][['fire_id', 'fire_zone', 'n_fall_hotspots', 'n_spring_hotspots',
           'hotspot_distance_m', 'dormancy_days', 'days_after_sdd',
           'dist_to_prev_perimeter_m', 'last_fall_date', 'first_spring_date',
           'first_spring_lat', 'first_spring_lon']].copy()
        ow_23_24 = ow_23_24.rename(columns={
            'fire_id':                  'fire_id_2023',
            'n_fall_hotspots':          'n_fall_hotspots_2023',
            'n_spring_hotspots':        'n_spring_hotspots_2024',
            'hotspot_distance_m':       'hotspot_distance_m_2324',
            'dormancy_days':            'dormancy_days_2324',
            'days_after_sdd':           'days_after_sdd_2324',
            'dist_to_prev_perimeter_m': 'dist_to_perimeter_2324',
            'last_fall_date':           'last_fall_date_2023',
            'first_spring_date':        'first_spring_date_2024'
        })

        # --- step 2: 2024 fires with spring 2025 hotspots in 2024 buffer ---
        ow_24_25 = df[
            (df['fire_year'] == 2024) &
            (df['overwintered'] == True) &
            (df['spring_year'] == 2025)
        ][['fire_id', 'n_fall_hotspots', 'n_spring_hotspots',
           'hotspot_distance_m', 'dormancy_days', 'days_after_sdd',
           'dist_to_prev_perimeter_m', 'last_fall_date', 'first_spring_date',
           'first_spring_lat', 'first_spring_lon']].copy()
        ow_24_25 = ow_24_25.rename(columns={
            'fire_id':                  'fire_id_2024',
            'n_fall_hotspots':          'n_fall_hotspots_2024',
            'n_spring_hotspots':        'n_spring_hotspots_2025',
            'hotspot_distance_m':       'hotspot_distance_m_2425',
            'dormancy_days':            'dormancy_days_2425',
            'days_after_sdd':           'days_after_sdd_2425',
            'dist_to_prev_perimeter_m': 'dist_to_perimeter_2425',
            'last_fall_date':           'last_fall_date_2024',
            'first_spring_date':        'first_spring_date_2025'
        })

        print(f"\nBuffer {dist}m:")
        print(f"  2023 fires overwintered into 2024: {len(ow_23_24)}")
        print(f"  2024 fires overwintered into 2025: {len(ow_24_25)}")

        if ow_23_24.empty or ow_24_25.empty:
            print(f"  No multi-year candidates possible")
            multiyear_results[dist] = pd.DataFrame()
            continue

        # --- step 3: match spring 2024 hotspot to 2024 perimeter ---
        # this links the 2023 overwintering event to a specific 2024 fire
        ow_23_24_gdf = gpd.GeoDataFrame(
            ow_23_24,
            geometry=gpd.points_from_xy(
                ow_23_24['first_spring_lon'],
                ow_23_24['first_spring_lat']
            ),
            crs='EPSG:4326'
        ).to_crs('EPSG:3005')
        ow_23_24_gdf['geometry'] = ow_23_24_gdf['geometry'].buffer(hotspot_buffer_m)

        perims_2024 = fires[fires['fire_year'] == 2024.0][['fire_id', 'geometry']].rename(
            columns={'fire_id': 'fire_id_2024'}
        )

        hotspot_in_2024_perim = match_hotspot_to_nearest_perimeter(
        ow_23_24_gdf[['fire_id_2023', 'geometry']].assign(fire_id_2023=ow_23_24_gdf['fire_id_2023']),
        perims_2024,
        id_col_left='fire_id_2023',
        id_col_right='fire_id_2024',
        max_dist_m=500
)
        print(f"  Spring 2024 hotspots matched to 2024 perimeters: {len(hotspot_in_2024_perim)}")

        if hotspot_in_2024_perim.empty:
            print(f"  No 2024 perimeter matches found for spring 2024 hotspots")
            multiyear_results[dist] = pd.DataFrame()
            continue

        # --- step 4: match spring 2025 hotspot to 2025 perimeter ---
        # this links the 2024 overwintering event to a specific 2025 fire
        ow_24_25_gdf = gpd.GeoDataFrame(
            ow_24_25,
            geometry=gpd.points_from_xy(
                ow_24_25['first_spring_lon'],
                ow_24_25['first_spring_lat']
            ),
            crs='EPSG:4326'
        ).to_crs('EPSG:3005')
        ow_24_25_gdf['geometry'] = ow_24_25_gdf['geometry'].buffer(hotspot_buffer_m)

        perims_2025 = fires[fires['fire_year'] == 2025.0][['fire_id', 'geometry']].rename(
            columns={'fire_id': 'fire_id_2025'}
        )

        # spring 2025 hotspot -> nearest 2025 perimeter within 500m
        hotspot_in_2025_perim = match_hotspot_to_nearest_perimeter(
            ow_24_25_gdf[['fire_id_2024', 'geometry']].assign(fire_id_2024=ow_24_25_gdf['fire_id_2024']),
            perims_2025,
            id_col_left='fire_id_2024',
            id_col_right='fire_id_2025',
            max_dist_m=500
        )

        print(f"  Spring 2025 hotspots matched to 2025 perimeters: {len(hotspot_in_2025_perim)}")

        if hotspot_in_2025_perim.empty:
            print(f"  No 2025 perimeter matches found for spring 2025 hotspots")
            multiyear_results[dist] = pd.DataFrame()
            continue

        # aggregate 2025 fire ids into list per 2024 fire
        hotspot_in_2025_perim_agg = (hotspot_in_2025_perim
            .groupby('fire_id_2024')['fire_id_2025']
            .apply(list)
            .reset_index()
        )

        # --- step 5: build the full chain 2023 -> 2024 -> 2025 ---
        # link: spring 2024 hotspot in 2024 perimeter -> 2024 fire overwintered into 2025
        # -> spring 2025 hotspot in 2025 perimeter
        multiyear = (hotspot_in_2024_perim                          # fire_id_2023, fire_id_2024
            .merge(hotspot_in_2025_perim_agg, on='fire_id_2024', how='inner')   # adds fire_id_2025
            .merge(ow_23_24.drop(columns=['first_spring_lat', 'first_spring_lon']),
                   on='fire_id_2023', how='inner')                  # adds 2023->2024 metrics
            .merge(ow_24_25.drop(columns=['first_spring_lat', 'first_spring_lon']),
                   on='fire_id_2024', how='inner')                  # adds 2024->2025 metrics
        )

        multiyear['buffer_dist'] = dist
        multiyear_results[dist] = multiyear

        print(f"  Confirmed multi-year overwintering fires: {len(multiyear)}")
        print(multiyear[['fire_id_2023', 'fire_id_2024', 'fire_id_2025',
                          'fire_zone', 'dormancy_days_2324', 'dormancy_days_2425',
                          'days_after_sdd_2324', 'days_after_sdd_2425']].to_string())

    return multiyear_results

multiyear_results = find_multiyear_overwintering(all_results, fires, buffer_distances)

# save
for dist in buffer_distances:
    if not multiyear_results[dist].empty:
        multiyear_results[dist].to_csv(
            f"../data/analysis/multiyear_overwintering_{dist}m.csv", index=False
        )
        print(f"Saved multiyear_overwintering_{dist}m.csv")


Buffer 500m:
  2023 fires overwintered into 2024: 14
  2024 fires overwintered into 2025: 4
  Spring 2024 hotspots matched to 2024 perimeters: 7
  Spring 2025 hotspots matched to 2025 perimeters: 4
  Confirmed multi-year overwintering fires: 6
  fire_id_2023 fire_id_2024   fire_id_2025              fire_zone  dormancy_days_2324  dormancy_days_2425  days_after_sdd_2324  days_after_sdd_2425
0  2023_G90288  2024_G90228  [2025_G80320]  Fort Nelson Fire Zone               298.0               252.0            56.528890            13.239742
1  2023_G90628  2024_G90228  [2025_G80320]                   both               272.0               252.0            50.618965            13.239742
2  2023_G91313  2024_G90228  [2025_G80320]  Fort Nelson Fire Zone               269.0               252.0            46.262006            13.239742
3  2023_G91494  2024_G90228  [2025_G80320]  Fort Nelson Fire Zone               302.0               252.0            69.037260            13.239742
4  2023_G91534 

In [24]:
print(multiyear_results[1000])

  fire_id_2023 fire_id_2024  dist_to_perimeter   fire_id_2025  \
0  2023_G90288  2024_G90228                0.0  [2025_G80320]   
1  2023_G90628  2024_G90228                0.0  [2025_G80320]   
2  2023_G91313  2024_G90228                0.0  [2025_G80320]   
3  2023_G91494  2024_G90228                0.0  [2025_G80320]   
4  2023_G91534  2024_G90399                0.0  [2025_G90400]   
5  2023_G92498  2024_G90228                0.0  [2025_G80320]   
6  2023_G92940  2024_G90228                0.0  [2025_G80320]   
7  2023_G93291  2024_G90228                0.0  [2025_G80320]   

               fire_zone  n_fall_hotspots_2023  n_spring_hotspots_2024  \
0  Fort Nelson Fire Zone                  1227                     549   
1                   both                  2730                     210   
2  Fort Nelson Fire Zone                  2267                      28   
3  Fort Nelson Fire Zone                   256                      27   
4  Fort Nelson Fire Zone                   7

BONUS

In [66]:
def find_multiyear_overwintering(all_results, fires, buffer_distances):
    multiyear_results = {}

    for dist in buffer_distances:
        df = all_results[dist].copy()

        # 2023 fires with spring 2024 hotspots in 2023 buffer
        ow_23_24 = df[
            (df['fire_year'] == 2023) &
            (df['overwintered'] == True) &
            (df['spring_year'] == 2024)
        ][['fire_id', 'fire_zone', 'n_fall_hotspots', 'n_spring_hotspots',
           'hotspot_distance_m', 'dormancy_days', 'days_after_sdd',
           'dist_to_prev_perimeter_m', 'last_fall_date', 'first_spring_date']].copy()
        ow_23_24 = ow_23_24.rename(columns={
            'fire_id':                  'fire_id_2023',
            'n_fall_hotspots':          'n_fall_hotspots_2023',
            'n_spring_hotspots':        'n_spring_hotspots_2024',
            'hotspot_distance_m':       'hotspot_distance_m_2324',
            'dormancy_days':            'dormancy_days_2324',
            'days_after_sdd':           'days_after_sdd_2324',
            'dist_to_prev_perimeter_m': 'dist_to_perimeter_2324',
            'last_fall_date':           'last_fall_date_2023',
            'first_spring_date':        'first_spring_date_2024'
        })

        # 2024 fires with spring 2025 hotspots in 2024 buffer
        # keep first spring hotspot coordinates to find 2025 perimeter
        ow_24_25 = df[
            (df['fire_year'] == 2024) &
            (df['overwintered'] == True) &
            (df['spring_year'] == 2025)
        ][['fire_id', 'n_fall_hotspots', 'n_spring_hotspots',
           'hotspot_distance_m', 'dormancy_days', 'days_after_sdd',
           'dist_to_prev_perimeter_m', 'last_fall_date', 'first_spring_date',
           'first_spring_lat', 'first_spring_lon']].copy()
        ow_24_25 = ow_24_25.rename(columns={
            'fire_id':                  'fire_id_2024',
            'n_fall_hotspots':          'n_fall_hotspots_2024',
            'n_spring_hotspots':        'n_spring_hotspots_2025',
            'hotspot_distance_m':       'hotspot_distance_m_2425',
            'dormancy_days':            'dormancy_days_2425',
            'days_after_sdd':           'days_after_sdd_2425',
            'dist_to_prev_perimeter_m': 'dist_to_perimeter_2425',
            'last_fall_date':           'last_fall_date_2024',
            'first_spring_date':        'first_spring_date_2025'
        })

        print(f"\nBuffer {dist}m:")
        print(f"  2023 fires with spring 2024 hotspots in buffer: {len(ow_23_24)}")
        print(f"  2024 fires with spring 2025 hotspots in buffer: {len(ow_24_25)}")

        if ow_23_24.empty or ow_24_25.empty:
            print(f"  No multi-year candidates possible")
            multiyear_results[dist] = pd.DataFrame()
            continue

        # find which 2025 perimeter the first spring 2025 hotspot falls within
        perims_2025 = fires[fires['fire_year'] == 2025.0][['fire_id', 'geometry']].rename(
            columns={'fire_id': 'fire_id_2025'}
        )

        # convert first spring hotspot to geodataframe
        ow_24_25_gdf = gpd.GeoDataFrame(
            ow_24_25,
            geometry=gpd.points_from_xy(
                ow_24_25['first_spring_lon'],
                ow_24_25['first_spring_lat']
            ),
            crs='EPSG:4326'
        ).to_crs('EPSG:3005')

        # spatial join to find which 2025 perimeter each spring hotspot falls within
        hotspot_in_2025_perim = gpd.sjoin(
            ow_24_25_gdf[['fire_id_2024', 'geometry']],
            perims_2025[['fire_id_2025', 'geometry']],
            how='left',
            predicate='within'
        )[['fire_id_2024', 'fire_id_2025']]

        # aggregate in case hotspot falls within multiple perimeters
        hotspot_in_2025_perim = (hotspot_in_2025_perim
            .groupby('fire_id_2024')['fire_id_2025']
            .apply(lambda x: list(x.dropna()))
            .reset_index()
        )

        # join 2025 fire ids back to ow_24_25
        ow_24_25 = ow_24_25.merge(hotspot_in_2025_perim, on='fire_id_2024', how='left')

        # spatially link 2023 and 2024 overwintering fires
        perims_2023_ow = fires[
            fires['fire_id'].isin(ow_23_24['fire_id_2023'])
        ][['fire_id', 'geometry']].rename(columns={'fire_id': 'fire_id_2023'})

        perims_2024_ow = fires[
            fires['fire_id'].isin(ow_24_25['fire_id_2024'])
        ][['fire_id', 'geometry']].rename(columns={'fire_id': 'fire_id_2024'})

        # buffer 2023 perimeters and intersect with 2024 perimeters
        perims_2023_buf = perims_2023_ow.copy()
        perims_2023_buf['geometry'] = perims_2023_buf['geometry'].buffer(dist)

        spatial_links_23_24 = gpd.sjoin(
            perims_2023_buf[['fire_id_2023', 'geometry']],
            perims_2024_ow[['fire_id_2024', 'geometry']],
            how='inner',
            predicate='intersects'
        )[['fire_id_2023', 'fire_id_2024']]

        print(f"  Spatial links 2023->2024: {len(spatial_links_23_24)}")

        if spatial_links_23_24.empty:
            print(f"  No spatial links found between 2023 and 2024 fires")
            multiyear_results[dist] = pd.DataFrame()
            continue

        # join everything together
        multiyear = (spatial_links_23_24
            .merge(ow_23_24, on='fire_id_2023', how='inner')
            .merge(ow_24_25.drop(columns=['first_spring_lat', 'first_spring_lon']),
                   on='fire_id_2024', how='inner')
        )

        multiyear['buffer_dist'] = dist
        multiyear_results[dist] = multiyear

        print(f"  Confirmed multi-year overwintering fires: {len(multiyear)}")
        print(multiyear[['fire_id_2023', 'fire_id_2024', 'fire_id_2025',
                          'fire_zone', 'dormancy_days_2324', 'dormancy_days_2425',
                          'days_after_sdd_2324', 'days_after_sdd_2425']].to_string())

    return multiyear_results

multiyear_results = find_multiyear_overwintering(all_results, fires, buffer_distances)

# save
for dist in buffer_distances:
    if not multiyear_results[dist].empty:
        multiyear_results[dist].to_csv(
            f"../data/analysis/multiyear_overwintering_{dist}m.csv", index=False
        )
        print(f"Saved multiyear_overwintering_{dist}m.csv")


Buffer 500m:
  2023 fires with spring 2024 hotspots in buffer: 14
  2024 fires with spring 2025 hotspots in buffer: 4
  Spatial links 2023->2024: 10
  Confirmed multi-year overwintering fires: 10
  fire_id_2023 fire_id_2024   fire_id_2025              fire_zone  dormancy_days_2324  dormancy_days_2425  days_after_sdd_2324  days_after_sdd_2425
0  2023_G90288  2024_G90228             []  Fort Nelson Fire Zone               298.0               252.0            56.528890            13.239742
1  2023_G90288  2024_G90686  [2025_G90216]  Fort Nelson Fire Zone               298.0               258.0            56.528890            35.763450
2  2023_G90628  2024_G90228             []                   both               272.0               252.0            50.618965            13.239742
3  2023_G91313  2024_G90228             []  Fort Nelson Fire Zone               269.0               252.0            46.262006            13.239742
4  2023_G91494  2024_G90228             []  Fort Nelson Fire Zo

In [1]:
def find_multiyear_overwintering(all_results, fires, buffer_distances, hotspot_buffer_m=500):
    multiyear_results = {}

    for dist in buffer_distances:
        df = all_results[dist].copy()

        # 2023 fires with spring 2024 hotspots in 2023 buffer
        ow_23_24 = df[
            (df['fire_year'] == 2023) &
            (df['overwintered'] == True) &
            (df['spring_year'] == 2024)
        ][['fire_id', 'fire_zone', 'n_fall_hotspots', 'n_spring_hotspots',
           'hotspot_distance_m', 'dormancy_days', 'days_after_sdd',
           'dist_to_prev_perimeter_m', 'last_fall_date', 'first_spring_date']].copy()
        ow_23_24 = ow_23_24.rename(columns={
            'fire_id':                  'fire_id_2023',
            'n_fall_hotspots':          'n_fall_hotspots_2023',
            'n_spring_hotspots':        'n_spring_hotspots_2024',
            'hotspot_distance_m':       'hotspot_distance_m_2324',
            'dormancy_days':            'dormancy_days_2324',
            'days_after_sdd':           'days_after_sdd_2324',
            'dist_to_prev_perimeter_m': 'dist_to_perimeter_2324',
            'last_fall_date':           'last_fall_date_2023',
            'first_spring_date':        'first_spring_date_2024'
        })

        # 2024 fires with spring 2025 hotspots in 2024 buffer
        ow_24_25 = df[
            (df['fire_year'] == 2024) &
            (df['overwintered'] == True) &
            (df['spring_year'] == 2025)
        ][['fire_id', 'n_fall_hotspots', 'n_spring_hotspots',
           'hotspot_distance_m', 'dormancy_days', 'days_after_sdd',
           'dist_to_prev_perimeter_m', 'last_fall_date', 'first_spring_date',
           'first_spring_lat', 'first_spring_lon']].copy()
        ow_24_25 = ow_24_25.rename(columns={
            'fire_id':                  'fire_id_2024',
            'n_fall_hotspots':          'n_fall_hotspots_2024',
            'n_spring_hotspots':        'n_spring_hotspots_2025',
            'hotspot_distance_m':       'hotspot_distance_m_2425',
            'dormancy_days':            'dormancy_days_2425',
            'days_after_sdd':           'days_after_sdd_2425',
            'dist_to_prev_perimeter_m': 'dist_to_perimeter_2425',
            'last_fall_date':           'last_fall_date_2024',
            'first_spring_date':        'first_spring_date_2025'
        })

        print(f"\nBuffer {dist}m:")
        print(f"  2023 fires overwintered into 2024: {len(ow_23_24)}")
        print(f"  2024 fires overwintered into 2025 (before 2025 perimeter check): {len(ow_24_25)}")

        if ow_23_24.empty or ow_24_25.empty:
            print(f"  No multi-year candidates possible")
            multiyear_results[dist] = pd.DataFrame()
            continue

        # --- validate spring 2025 hotspot against 2025 perimeters ---
        perims_2025 = fires[fires['fire_year'] == 2025.0][['fire_id', 'geometry']].rename(
            columns={'fire_id': 'fire_id_2025'}
        )

        # convert first spring 2025 hotspot to projected points
        ow_24_25_gdf = gpd.GeoDataFrame(
            ow_24_25,
            geometry=gpd.points_from_xy(
                ow_24_25['first_spring_lon'],
                ow_24_25['first_spring_lat']
            ),
            crs='EPSG:4326'
        ).to_crs('EPSG:3005')

        # buffer hotspot by VIIRS pixel size to account for positional uncertainty
        ow_24_25_gdf_buf = ow_24_25_gdf.copy()
        ow_24_25_gdf_buf['geometry'] = ow_24_25_gdf_buf['geometry'].buffer(hotspot_buffer_m)

        # find which 2025 perimeter each spring hotspot intersects
        hotspot_in_2025_perim = gpd.sjoin(
            ow_24_25_gdf_buf[['fire_id_2024', 'geometry']],
            perims_2025[['fire_id_2025', 'geometry']],
            how='inner',  # inner join — only keep hotspots that match a 2025 perimeter
            predicate='intersects'
        )[['fire_id_2024', 'fire_id_2025']]

        # aggregate 2025 fire ids into list per 2024 fire
        hotspot_in_2025_perim_agg = (hotspot_in_2025_perim
            .groupby('fire_id_2024')['fire_id_2025']
            .apply(list)
            .reset_index()
        )

        print(f"  2024 fires whose 2025 hotspot matches a 2025 perimeter: {len(hotspot_in_2025_perim_agg)}")

        if hotspot_in_2025_perim_agg.empty:
            print(f"  No 2025 perimeter matches found")
            multiyear_results[dist] = pd.DataFrame()
            continue

        # filter ow_24_25 to only fires with a matched 2025 perimeter
        ow_24_25_validated = ow_24_25.merge(
            hotspot_in_2025_perim_agg, on='fire_id_2024', how='inner'
        ).drop(columns=['first_spring_lat', 'first_spring_lon'])

        print(f"  Validated 2024 overwintering fires with 2025 perimeter match: {len(ow_24_25_validated)}")

        # spatially link 2023 and validated 2024 overwintering fires
        perims_2023_ow = fires[
            fires['fire_id'].isin(ow_23_24['fire_id_2023'])
        ][['fire_id', 'geometry']].rename(columns={'fire_id': 'fire_id_2023'})

        perims_2024_ow = fires[
            fires['fire_id'].isin(ow_24_25_validated['fire_id_2024'])
        ][['fire_id', 'geometry']].rename(columns={'fire_id': 'fire_id_2024'})

        # buffer 2023 perimeters and intersect with 2024 perimeters
        perims_2023_buf = perims_2023_ow.copy()
        perims_2023_buf['geometry'] = perims_2023_buf['geometry'].buffer(dist)

        spatial_links_23_24 = gpd.sjoin(
            perims_2023_buf[['fire_id_2023', 'geometry']],
            perims_2024_ow[['fire_id_2024', 'geometry']],
            how='inner',
            predicate='intersects'
        )[['fire_id_2023', 'fire_id_2024']]

        print(f"  Spatial links 2023->2024: {len(spatial_links_23_24)}")

        if spatial_links_23_24.empty:
            print(f"  No spatial links found between 2023 and 2024 fires")
            multiyear_results[dist] = pd.DataFrame()
            continue

        # join all metrics together
        multiyear = (spatial_links_23_24
            .merge(ow_23_24, on='fire_id_2023', how='inner')
            .merge(ow_24_25_validated, on='fire_id_2024', how='inner')
        )

        multiyear['buffer_dist'] = dist
        multiyear_results[dist] = multiyear

        print(f"  Confirmed multi-year overwintering fires: {len(multiyear)}")
        print(multiyear[['fire_id_2023', 'fire_id_2024', 'fire_id_2025',
                          'fire_zone', 'dormancy_days_2324', 'dormancy_days_2425',
                          'days_after_sdd_2324', 'days_after_sdd_2425']].to_string())

    return multiyear_results

multiyear_results = find_multiyear_overwintering(all_results, fires, buffer_distances)

# save
for dist in buffer_distances:
    if not multiyear_results[dist].empty:
        multiyear_results[dist].to_csv(
            f"../data/analysis/multiyear_overwintering_{dist}m.csv", index=False
        )
        print(f"Saved multiyear_overwintering_{dist}m.csv")

NameError: name 'all_results' is not defined